In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import mne
from scipy.io import loadmat
import sys
import Original_Code.old_act as act_cpu_lib

sys.path.append("../..")
import act as act_cpu_lib_2

In [2]:
num_epochs = 50
order = 10

In [ ]:
import time
import numpy as np

def run_act_benchmark(data, FS, epoch_length, tc_info, fc_info,
                      logDt_info, c_info, order,
                      num_epochs=1, channels=None, name="DATA"):

    # Storage
    gen_time_list, gen_time_list_new = [], []
    runtime_per_epoch, runtime_per_epoch_new = [], []
    all_order_runtime, all_order_runtime_new = [], []
    full_runtime, full_runtime_new = [], []
    mse_list, mse_list_new = [], []
    norm_residue, norm_residue_new = [], []

    # Ensure data is 2D (samples, channels)
    if data.ndim == 1:
        data = data[:, np.newaxis]

    if channels is None:
        channels = list(range(data.shape[1]))

    for i in range(5):
        # -------- OLD ACT --------
        start_gen_time = time.perf_counter()
        act_old = act_cpu_lib.ACT(FS=FS, length=epoch_length,
                          tc_info=tc_info, fc_info=fc_info,
                          logDt_info=logDt_info, c_info=c_info,
                          force_regenerate=True, mute=True)
        gen_time = time.perf_counter() - start_gen_time
        gen_time_list.append(gen_time)

        start_runtime = time.perf_counter()

        for epoch in range(num_epochs):
            start_idx = epoch * epoch_length
            end_idx = start_idx + epoch_length

            start_epoch = time.perf_counter()

            if name == "FMCW":
                segment = data[epoch, :]
                out = act_old.transform(segment, order=order, debug=False)

                mse_list.append(out["mse"])
                norm_residue.append(out["norm_residue"])
            else:
                for ch_idx in range(len(channels)):
                    segment = data[start_idx:end_idx, ch_idx]
                    out = act_old.transform(segment, order=order, debug=False)

                    mse_list.append(out["mse"])
                    norm_residue.append(out["norm_residue"])

            runtime_per_epoch.append(time.perf_counter() - start_epoch)

        runtime = time.perf_counter() - start_runtime
        all_order_runtime.append(runtime)
        full_runtime.append(runtime + gen_time)

        # -------- NEW ACT --------
        start_gen_time = time.perf_counter()
        act_new = act_cpu_lib_2.ACT(FS=FS, length=epoch_length,
                            tc_info=tc_info, fc_info=fc_info,
                            logDt_info=logDt_info, c_info=c_info,
                            force_regenerate=True, mute=True)
        gen_time_new = time.perf_counter() - start_gen_time
        gen_time_list_new.append(gen_time_new)

        start_runtime = time.perf_counter()

        for epoch in range(num_epochs):
            start_idx = epoch * epoch_length
            end_idx = start_idx + epoch_length

            start_epoch = time.perf_counter()

            if name == "FMCW":
                segment = data[epoch, :]
                out = act_new.transform(segment, order=order, debug=False)

                mse_list_new.append(out["mse"])
                norm_residue_new.append(out["norm_residue"])
            else:
                for ch_idx in range(len(channels)):
                    segment = data[start_idx:end_idx, ch_idx]
                    out = act_new.transform(segment, order=order, debug=False)

                    mse_list_new.append(out["mse"])
                    norm_residue_new.append(out["norm_residue"])

            runtime_per_epoch_new.append(time.perf_counter() - start_epoch)

        runtime_new = time.perf_counter() - start_runtime
        all_order_runtime_new.append(runtime_new)
        full_runtime_new.append(runtime_new + gen_time_new)

        print("New runtime done")

    # -------- REPORT --------
    print(f"\n===== {name} RESULTS =====")

    print(f"Old gen: {np.mean(gen_time_list):.4f} ± {np.std(gen_time_list):.4f}")
    print(f"New gen: {np.mean(gen_time_list_new):.4f} ± {np.std(gen_time_list_new):.4f}")

    print(f"Old epoch: {np.mean(runtime_per_epoch):.4f} ± {np.std(runtime_per_epoch):.4f}")
    print(f"New epoch: {np.mean(runtime_per_epoch_new):.4f} ± {np.std(runtime_per_epoch_new):.4f}")

    print(f"Old runtime: {np.mean(all_order_runtime):.4f} ± {np.std(all_order_runtime):.4f}")
    print(f"New runtime: {np.mean(all_order_runtime_new):.4f} ± {np.std(all_order_runtime_new):.4f}")

    print(f"Old full: {np.mean(full_runtime):.4f} ± {np.std(full_runtime):.4f}")
    print(f"New full: {np.mean(full_runtime_new):.4f} ± {np.std(full_runtime_new):.4f}")

    print(f"Old MSE: {np.mean(mse_list):.4f} ± {np.std(mse_list):.4f}")
    print(f"New MSE: {np.mean(mse_list_new):.4f} ± {np.std(mse_list_new):.4f}")

    print(f"Old residue: {np.mean(norm_residue):.4f} ± {np.std(norm_residue):.4f}")
    print(f"New residue: {np.mean(norm_residue_new):.4f} ± {np.std(norm_residue_new):.4f}")

In [ ]:
FS = 256 / 0.015
lowcut = 10
highcut = int(FS/2)
tc_info = (0, 256, 16)
fc_info = (lowcut, highcut, 100)
logDt_info = (-4, 0, 0.3)
c_info = (-10, 10, 0.75)

data = np.load("../Data/data_SAAB_SIRS_77GHz_FMCW.npy", allow_pickle=True)
all_segments = data[0, 1][512:768, :]                     # (256, 1228)
all_segments = np.abs(all_segments).astype(np.float32).T  # (1228, 256)

b, a = butter(4, [lowcut/(FS/2), highcut/(FS/2)], btype='band')
for i in range(all_segments.shape[0]):
    all_segments[i] = filtfilt(b, a, all_segments[i])

In [5]:
fmcw_data = all_segments
with np.errstate(all='ignore'):
    run_act_benchmark(
        fmcw_data,
        FS=FS,
        epoch_length=256,
        tc_info=tc_info,
        fc_info=fc_info,
        logDt_info=logDt_info,
        c_info=c_info,
        order=order,
        num_epochs=num_epochs,
        name="FMCW"
    )

Old gen done
Old runtime done
Dictionary length: 520128
New gen done
New runtime done
Old gen done
Old runtime done
Dictionary length: 520128
New gen done
New runtime done
Old gen done
Old runtime done
Dictionary length: 520128
New gen done
New runtime done
Old gen done
Old runtime done
Dictionary length: 520128
New gen done
New runtime done
Old gen done
Old runtime done
Dictionary length: 520128
New gen done
New runtime done

===== FMCW RESULTS =====
Old gen: 8.7781 ± 0.0728
New gen: 9.6780 ± 0.0631
Old epoch: 0.3877 ± 0.1427
New epoch: 0.3005 ± 0.0396
Old runtime: 19.3853 ± 2.5518
New runtime: 15.0230 ± 0.2353
Old full: 28.1634 ± 2.5597
New full: 24.7010 ± 0.2277
Old MSE: 0.8921 ± 0.0088
New MSE: 0.0238 ± 0.0223
Old residue: 0.9445 ± 0.0047
New residue: 0.1439 ± 0.0554


In [ ]:
FS = 256
epoch_length = FS
channels = ['HB_1']
tc_info = (0, epoch_length, 16)
fc_info = (0.5, 15, 0.5)
logDt_info = (-4, 1, 0.5)
c_info = (-10, 10, 0.5)

data_file = os.path.join(os.getcwd(), '../Data/sub-1_task-Sleep_acq-headband_eeg.edf')
raw = mne.io.read_raw_edf(data_file, preload=True, verbose=False)
raw.pick_channels(channels)
raw.notch_filter(50, verbose=False)
raw.filter(0.1,20, verbose=False)
eeg = raw.get_data().T.astype(np.float32)

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


In [7]:
eeg_data = eeg
run_act_benchmark(
    eeg_data,
    FS=256,
    epoch_length=256,
    tc_info=tc_info,
    fc_info=fc_info,
    logDt_info=logDt_info,
    c_info=c_info,
    order=order,
    num_epochs=num_epochs,
    name="EEG"
)

Old gen done


/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encoun

Old runtime done
Dictionary length: 185600
New gen done
New runtime done
Old gen done


/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encoun

Old runtime done
Dictionary length: 185600
New gen done
New runtime done
Old gen done


...

Old runtime done
Dictionary length: 185600
New gen done
New runtime done
Old gen done


/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encoun

Old runtime done
Dictionary length: 185600
New gen done
New runtime done
Old gen done


/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encountered in exp
  Dt = np.exp(logDt) # Calculating the non-log Delta_t value
/Users/nishant/Documents/GitHub/acttesting/Testing Scripts/../act.py:101: RuntimeWarning: overflow encoun

Old runtime done
Dictionary length: 185600
New gen done
New runtime done

===== EEG RESULTS =====
Old gen: 2.9426 ± 0.0295
New gen: 3.3003 ± 0.0191
Old epoch: 0.2033 ± 0.0435
New epoch: 0.0446 ± 0.0114
Old runtime: 10.1647 ± 0.5163
New runtime: 2.2304 ± 0.0802
Old full: 13.1073 ± 0.4929
New full: 5.5307 ± 0.0909
Old MSE: 3.0407 ± 4.2190
New MSE: 0.0011 ± 0.0009
Old residue: 1.4166 ± 1.0168
New residue: 0.0305 ± 0.0115


In [ ]:
from scipy.io import loadmat
from scipy.signal import iirnotch
FS = 2000
epoch_length = 200
tc_info = (0,200,32)
fc_info = (20,450,10)
logDt_info = (-4,0,0.3)
c_info = (-10,10,0.75)

mat = loadmat('../Data/S2_E3_A1_basic_movement.mat')
# print(mat.keys())
emg = mat['emg']

for freq in [50, 100, 150, 200, 250, 300, 350, 400, 450]:
    b_n, a_n = iirnotch(freq, Q=30, fs=FS)
    emg = filtfilt(b_n, a_n, emg, axis=0)
    
b, a = butter(4, [20/(FS/2), 450/(FS/2)], btype='band')
emg_filtered = np.zeros_like(emg)
for ch in range(emg.shape[1]):
    emg_filtered[:, ch] = filtfilt(b, a, emg[:, ch])
emg = emg_filtered[:,0]

In [9]:
emg = emg
with np.errstate(all='ignore'):
    run_act_benchmark(
        emg,   # shape (samples, channels)
        FS=2000,
        epoch_length=200,
        tc_info=(0,200,32),
        fc_info=(20,450,10),
        logDt_info=(-4,0,0.3),
        c_info=(-10,10,0.75),
        order=order,
        num_epochs=num_epochs,
        channels=None,
        name="EMG"
    )

Old gen done
Old runtime done
Dictionary length: 113778
New gen done
New runtime done
Old gen done
Old runtime done
Dictionary length: 113778
New gen done
New runtime done
Old gen done
Old runtime done
Dictionary length: 113778
New gen done
New runtime done
Old gen done
Old runtime done
Dictionary length: 113778
New gen done
New runtime done
Old gen done
Old runtime done
Dictionary length: 113778
New gen done
New runtime done

===== EMG RESULTS =====
Old gen: 1.7099 ± 0.0399
New gen: 1.9328 ± 0.0177
Old epoch: 0.5965 ± 0.0565
New epoch: 0.2734 ± 0.0279
Old runtime: 29.8271 ± 0.8407
New runtime: 13.6719 ± 0.4328
Old full: 31.5370 ± 0.8692
New full: 15.6047 ± 0.4492
Old MSE: 0.8612 ± 0.0341
New MSE: 0.2026 ± 0.0411
Old residue: 0.9278 ± 0.0186
New residue: 0.4478 ± 0.0454
